# The Encoder--Decoder Architecture
:label:`sec_encoder-decoder`

In general sequence-to-sequence problems
like machine translation
(:numref:`sec_machine_translation`),
inputs and outputs are of varying lengths
that are unaligned.
The standard approach to handling this sort of data
is to design an *encoder--decoder* architecture (:numref:`fig_encoder_decoder`)
consisting of two major components:
an *encoder* that takes a variable-length sequence as input,
and a *decoder* that acts as a conditional language model,
taking in the encoded input
and the leftwards context of the target sequence
and predicting the subsequent token in the target sequence.


![The encoder--decoder architecture.](../img/encoder-decoder.svg)
:label:`fig_encoder_decoder`

Let's take machine translation from English to French as an example.
Given an input sequence in English:
"They", "are", "watching", ".",
this encoder--decoder architecture
first encodes the variable-length input into a state,
then decodes the state
to generate the translated sequence,
token by token, as output:
"Ils", "regardent", ".".
Since the encoder--decoder architecture
forms the basis of different sequence-to-sequence models
in subsequent sections,
this section will convert this architecture
into an interface that will be implemented later.


In [1]:
from torch import nn
from d2l import torch as d2l

## (**Encoder**)

In the encoder interface,
we just specify that
the encoder takes variable-length sequences as input `X`.
The implementation will be provided
by any model that inherits this base `Encoder` class.


In [2]:
class Encoder(nn.Module):  #@save
    """The base encoder interface for the encoder--decoder architecture."""
    def __init__(self):
        super().__init__()

    # Later there can be additional arguments (e.g., length excluding padding)
    def forward(self, X, *args):
        raise NotImplementedError

## [**Decoder**]

In the following decoder interface,
we add an additional `init_state` method
to convert the encoder output (`enc_all_outputs`)
into the encoded state.
Note that this step
may require extra inputs,
such as the valid length of the input,
which was explained
in :numref:`sec_machine_translation`.
To generate a variable-length sequence token by token,
every time the decoder may map an input
(e.g., the generated token at the previous time step)
and the encoded state
into an output token at the current time step.


In [3]:
class Decoder(nn.Module):  #@save
    """The base decoder interface for the encoder--decoder architecture."""
    def __init__(self):
        super().__init__()

    # Later there can be additional arguments (e.g., length excluding padding)
    def init_state(self, enc_all_outputs, *args):
        raise NotImplementedError

    def forward(self, X, state):
        raise NotImplementedError

## [**Putting the Encoder and Decoder Together**]

In the forward propagation,
the output of the encoder
is used to produce the encoded state,
and this state will be further used
by the decoder as one of its input.


In [4]:
class EncoderDecoder(d2l.Classifier):  #@save
    """The base class for the encoder--decoder architecture."""
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_all_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_all_outputs, *args)
        # Return decoder output only
        return self.decoder(dec_X, dec_state)[0]

In the next section,
we will see how to apply RNNs to design
sequence-to-sequence models based on
this encoder--decoder architecture.


## Summary

Encoder-decoder architectures
can handle inputs and outputs
that both consist of variable-length sequences
and thus are suitable for sequence-to-sequence problems
such as machine translation.
The encoder takes a variable-length sequence as input
and transforms it into a state with a fixed shape.
The decoder maps the encoded state of a fixed shape
to a variable-length sequence.


## Exercises

1. Suppose that we use neural networks to implement the encoder--decoder architecture. Do the encoder and the decoder have to be the same type of neural network?
1. Besides machine translation, can you think of another application where the encoder--decoder architecture can be applied?


[Discussions](https://discuss.d2l.ai/t/1061)


In [1]:
I'll address both questions about encoder-decoder architectures in neural networks.

```markdown
# Encoder-Decoder Architecture Flexibility and Applications

## Question 1: Do the encoder and decoder have to be the same type of neural network?

**No, the encoder and decoder do not have to be the same type of neural network.** In fact, heterogeneous encoder-decoder architectures are quite common and often beneficial for specific tasks.

### Key Insights:

1. **Architectural Independence**
   - The encoder converts input sequences into a context vector (or a sequence of vectors)
   - The decoder generates output sequences from this representation
   - These functions are separate and can be optimized independently

2. **Common Heterogeneous Combinations**:
   - **CNN encoder + RNN decoder**: 
     - The CNN efficiently extracts spatial features from inputs like images
     - The RNN generates sequential outputs like captions
     - Example application: Image captioning (CNN encodes image, LSTM generates text)

   - **Transformer encoder + RNN decoder**:
     - Transformer better captures long-range dependencies in the source text
     - RNN provides auto-regressive generation with potentially lower computational cost
     - Example application: Some earlier neural machine translation systems

   - **RNN encoder + Transformer decoder**:
     - RNN processes sequential input efficiently
     - Transformer decoder leverages attention for flexible generation
     - Example application: Speech recognition systems

3. **Technical Compatibility Requirements**:
   - The only strict requirement is dimensional compatibility between:
     - The encoder's output representation
     - The decoder's expected input format
   - This can be addressed with linear projection layers if needed

4. **Implementation Example**:
   ```python
   class HybridEncoderDecoder(nn.Module):
       def __init__(self, input_dim, hidden_dim, output_dim):
           super().__init__()
           # CNN-based encoder
           self.encoder = nn.Sequential(
               nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
               nn.ReLU(),
               nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
               nn.AdaptiveAvgPool1d(1)  # Global pooling to fixed dimension
           )
           
           # RNN-based decoder
           self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
           self.output_layer = nn.Linear(hidden_dim, output_dim)
           
       def forward(self, x, target_len):
           # Encode input
           x = x.permute(0, 2, 1)  # Reshape for CNN
           context = self.encoder(x).squeeze(-1)
           
           # Initialize decoder input and hidden state
           decoder_input = context.unsqueeze(1)  # Add sequence dimension
           hidden = (context.unsqueeze(0), context.unsqueeze(0))  # For LSTM
           
           outputs = []
           for t in range(target_len):
               output, hidden = self.decoder(decoder_input, hidden)
               prediction = self.output_layer(output)
               outputs.append(prediction)
               decoder_input = output  # Use output as next input
               
           return torch.cat(outputs, dim=1)
   ```

5. **Performance Considerations**:
   - Different neural network architectures have different strengths
   - Choosing complementary architectures can improve performance
   - The encoder should match the input data structure
   - The decoder should match the output generation requirements

## Question 2: Other applications of encoder-decoder architectures

Encoder-decoder architectures extend far beyond machine translation. Here are several important applications:

### 1. Text Summarization
- **Encoder**: Processes the full document
- **Decoder**: Generates a concise summary
- **Example architecture**: BART, PEGASUS, or T5 models
- **Key feature**: The encoder needs to capture document-level semantics, while the decoder needs to generate fluent, coherent summaries

### 2. Image/Video Captioning
- **Encoder**: CNN or Vision Transformer processing visual input
- **Decoder**: RNN or Transformer generating natural language descriptions
- **Implementation example**:
  ```python
  class ImageCaptioningModel(nn.Module):
      def __init__(self, vocab_size):
          super().__init__()
          # ResNet encoder
          self.encoder = models.resnet50(pretrained=True)
          self.encoder.fc = nn.Linear(2048, 512)  # Project to embedding dim
          
          # LSTM decoder
          self.embedding = nn.Embedding(vocab_size, 512)
          self.decoder = nn.LSTM(512, 512)
          self.output = nn.Linear(512, vocab_size)
          
      def forward(self, image, captions=None, max_len=20):
          # Encode image
          features = self.encoder(image).unsqueeze(1)  # [B, 1, 512]
          
          if self.training:
              # Teacher forcing during training
              word_embeddings = self.embedding(captions)  # [B, seq_len, 512]
              decoder_input = torch.cat([features, word_embeddings[:, :-1]], dim=1)
              outputs, _ = self.decoder(decoder_input)
              predictions = self.output(outputs)
              
          else:
              # Autoregressive generation during inference
              current_input = features
              predictions = []
              hidden = None
              
              for t in range(max_len):
                  output, hidden = self.decoder(current_input, hidden)
                  prediction = self.output(output)
                  predictions.append(prediction)
                  
                  # Get word embedding for next input
                  predicted_token = prediction.argmax(dim=-1)
                  current_input = self.embedding(predicted_token)
                  
              predictions = torch.cat(predictions, dim=1)
              
          return predictions
  ```

### 3. Speech Recognition
- **Encoder**: Processes audio signals (typically using convolutional or recurrent layers)
- **Decoder**: Generates text transcriptions
- **Example**: Listen, Attend and Spell (LAS) architecture
- **Key challenge**: Handling variable-length inputs and outputs with different modalities

### 4. Code Generation
- **Encoder**: Processes natural language requirements or code context
- **Decoder**: Generates programming code
- **Example**: GitHub Copilot, based on the Codex model
- **Unique aspect**: Requires understanding of both natural language and programming language syntax

### 5. Data-to-Text Generation
- **Encoder**: Processes structured data (tables, graphs, knowledge bases)
- **Decoder**: Produces natural language descriptions
- **Applications**: Generating reports from financial data, sports commentary from game statistics
- **Example implementation**:
  ```python
  class TableToTextGenerator(nn.Module):
      def __init__(self, input_dim, hidden_dim, vocab_size):
          super().__init__()
          # Table encoder (processes tabular data)
          self.field_embeddings = nn.Embedding(num_fields, hidden_dim)
          self.value_encoder = nn.LSTM(input_dim, hidden_dim)
          
          # Text decoder
          self.decoder = nn.TransformerDecoder(
              nn.TransformerDecoderLayer(hidden_dim, nhead=8), 
              num_layers=6
          )
          self.word_embedding = nn.Embedding(vocab_size, hidden_dim)
          self.output_layer = nn.Linear(hidden_dim, vocab_size)
  ```

### 6. Question Answering
- **Encoder**: Processes context document and question
- **Decoder**: Generates the answer
- **Example**: T5 model for generative QA
- **Key design**: Using cross-attention to focus on relevant parts of the document

### 7. Style Transfer
- **Encoder**: Captures content from source text
- **Decoder**: Regenerates content in target style
- **Application**: Formal to informal language conversion, Shakespeare to modern English

### 8. Music Generation
- **Encoder**: Processes musical context or conditions
- **Decoder**: Generates new musical sequences
- **Example**: MusicTransformer architecture
- **Unique aspect**: Handling multi-track polyphonic music representation

### 9. Video Prediction
- **Encoder**: Processes past video frames
- **Decoder**: Generates future video frames
- **Key challenge**: Handling spatial and temporal relationships simultaneously

### 10. 3D Shape Completion
- **Encoder**: Processes partial 3D data (point clouds or voxels)
- **Decoder**: Generates complete 3D representations
- **Application**: Robotics, AR/VR, computer graphics

The versatility of encoder-decoder architectures stems from their ability to transform between different data representations and their inherent sequence-to-sequence modeling capability, making them applicable whenever we need to map between structured inputs and outputs of varying types and lengths.
```

SyntaxError: unterminated string literal (detected at line 1) (1290419477.py, line 1)